# 🏪 Market Businessman Desktop App - POS + Inventory Manager

A lightweight, full-featured **Point of Sale (POS) and Inventory Management System** built with **Python**, **Tkinter + ttk**, and **SQLite3**.

---

### 📋 System Overview & Architecture

| Screen / Module | Main Features |
| :--- | :--- |
| **Dashboard** | Today’s sales revenue, total units sold, low-stock warnings, recent sales activity, quick restock button |
| **Products Catalog** | Add, edit, delete, and search products with SKU/barcode, cost price, selling price, current stock, and min stock alert levels |
| **Billing / POS** | Barcode / SKU quick-scan, product search catalog, dynamic cart management, customer selection, discount %, GST %, grand total calculation, invoice generation |
| **Inventory Manager** | Manual Stock In (purchase) and Stock Out (damage/return), complete audit log history, automatic stock deduction on checkout |
| **Customers** | Customer directory with phone lookup and full purchase history per customer |
| **Reports & Analytics** | Daily/weekly sales summary, Cost of Goods Sold (COGS), estimated gross profit, top-selling products, and revenue trend chart |
| **Role Management** | Toggle between **Admin/Owner** (full access) and **Cashier** (cost prices and profit margins hidden) |

---

### 🛠️ Technology Stack
- **Python 3**
- **Tkinter & ttk**: Standard built-in Python GUI library with modern themed widgets
- **SQLite3**: Built-in zero-configuration ACID relational database
- **Optional Enhancements**:
  - `matplotlib` for interactive daily sales charts
  - `reportlab` for generating formatted PDF invoices (falls back cleanly to `.txt` invoices if not installed)

## 📦 Step 1: Install Optional Packages & Verify Environment

Run the cell below to check for optional packages (`matplotlib` and `reportlab`). The application runs smoothly with built-in libraries alone, but these optional packages unlock visual revenue charts and PDF invoice export.

In [ ]:
import sys
import os
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox, simpledialog, filedialog
from datetime import datetime, timedelta

# Check optional dependencies
MATPLOTLIB_AVAILABLE = False
try:
    import matplotlib
    matplotlib.use("TkAgg")
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
    MATPLOTLIB_AVAILABLE = True
    print("✅ matplotlib is installed and ready for visual analytics charts.")
except ImportError:
    print("ℹ️  matplotlib is not installed. Tabular sales breakdown will be used.")
    print("   To enable sales graphs, run: pip install matplotlib")

REPORTLAB_AVAILABLE = False
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
    REPORTLAB_AVAILABLE = True
    print("✅ reportlab is installed and ready for PDF invoice generation.")
except ImportError:
    print("ℹ️  reportlab is not installed. Formatted text receipts (.txt) will be generated.")
    print("   To enable PDF invoices, run: pip install reportlab")

## 🗄️ Step 2: Database Setup & Relational Schema

Here we create the SQLite database schema (`market.db`) containing five interconnected tables:
1. **`products`**: Product catalog with SKU, name, category, cost price, selling price, current stock, and low-stock threshold.
2. **`customers`**: Customer database with phone numbers, emails, and registration date.
3. **`sales`**: Invoices table tracking totals, discounts, taxes (GST), payment method, and timestamps.
4. **`sale_items`**: Line items per invoice, recording unit cost, unit selling price, and quantity sold.
5. **`inventory_logs`**: Complete audit trail of stock adjustments (Stock In, Stock Out, Sales, Audit corrections).

In [ ]:
DB_NAME = "market.db"
INVOICE_DIR = "invoices"

def init_db(db_path=DB_NAME):
    """Initialize SQLite database schema."""
    os.makedirs(INVOICE_DIR, exist_ok=True)
    con = sqlite3.connect(db_path)
    cur = con.cursor()

    # Products Table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            sku TEXT UNIQUE NOT NULL,
            name TEXT NOT NULL,
            category TEXT DEFAULT 'General',
            cost_price REAL NOT NULL DEFAULT 0.0,
            selling_price REAL NOT NULL,
            stock INTEGER NOT NULL DEFAULT 0,
            min_stock INTEGER NOT NULL DEFAULT 5
        )
    """)

    # Customers Table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            phone TEXT UNIQUE NOT NULL,
            email TEXT,
            created_at TEXT NOT NULL
        )
    """)

    # Sales (Invoices) Table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS sales (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            invoice_no TEXT UNIQUE NOT NULL,
            customer_id INTEGER,
            customer_name TEXT,
            customer_phone TEXT,
            subtotal REAL NOT NULL,
            discount_pct REAL DEFAULT 0.0,
            discount_amt REAL DEFAULT 0.0,
            gst_pct REAL DEFAULT 0.0,
            gst_amt REAL DEFAULT 0.0,
            grand_total REAL NOT NULL,
            payment_method TEXT DEFAULT 'Cash',
            cashier TEXT DEFAULT 'Admin',
            sold_at TEXT NOT NULL
        )
    """)

    # Sale Items Table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS sale_items (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            sale_id INTEGER NOT NULL,
            product_id INTEGER NOT NULL,
            sku TEXT,
            product_name TEXT NOT NULL,
            cost_price REAL NOT NULL,
            unit_price REAL NOT NULL,
            qty INTEGER NOT NULL,
            line_total REAL NOT NULL,
            FOREIGN KEY (sale_id) REFERENCES sales(id)
        )
    """)

    # Inventory Audit Logs Table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS inventory_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            product_id INTEGER NOT NULL,
            product_name TEXT NOT NULL,
            change_type TEXT NOT NULL,
            qty_change INTEGER NOT NULL,
            previous_stock INTEGER,
            new_stock INTEGER,
            reason TEXT,
            logged_at TEXT NOT NULL
        )
    """)

    con.commit()
    con.close()
    print(f"Database schema initialized successfully at '{db_path}'.")

init_db()

## 🌾 Step 3: Populate Sample Retail Store Demo Data

To make the application immediately interactive, the function below populates realistic grocery and retail products (including items with low stock to trigger dashboard alerts), sample customers, and demo sales for reporting analytics.

In [ ]:
def seed_sample_data(db_path=DB_NAME):
    """Seed initial demo data if database tables are empty."""
    con = sqlite3.connect(db_path)
    cur = con.cursor()

    # 1. Seed Products if empty
    cur.execute("SELECT COUNT(*) FROM products")
    if cur.fetchone()[0] == 0:
        sample_products = [
            ("SKU-1001", "Basmati Rice 1kg", "Grains & Rice", 75.0, 95.0, 30, 8),
            ("SKU-1002", "Refined Sunflower Oil 1L", "Oils", 110.0, 135.0, 20, 6),
            ("SKU-1003", "Whole Wheat Flour 5kg", "Flour & Atta", 185.0, 220.0, 15, 5),
            ("SKU-1004", "Pure White Sugar 1kg", "Sweeteners", 38.0, 48.0, 50, 10),
            ("SKU-1005", "Assam Gold Tea 250g", "Beverages", 95.0, 125.0, 4, 6),    # Low stock alert
            ("SKU-1006", "Herbal Bathing Soap 125g", "Personal Care", 28.0, 38.0, 3, 8), # Low stock alert
            ("SKU-1007", "Milk Sandwich Bread 400g", "Bakery", 30.0, 42.0, 12, 5),
            ("SKU-1008", "Dark Chocolate Slab 100g", "Confectionery", 65.0, 85.0, 25, 6),
            ("SKU-1009", "Roasted Ground Coffee 200g", "Beverages", 140.0, 180.0, 18, 5),
            ("SKU-1010", "Dishwash Gel 500ml", "Household", 70.0, 95.0, 2, 5)         # Low stock alert
        ]
        cur.executemany("""
            INSERT INTO products (sku, name, category, cost_price, selling_price, stock, min_stock)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, sample_products)

        # Record initial inventory logs
        now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cur.execute("SELECT id, name, stock FROM products")
        for pid, pname, pstock in cur.fetchall():
            cur.execute("""
                INSERT INTO inventory_logs (product_id, product_name, change_type, qty_change, previous_stock, new_stock, reason, logged_at)
                VALUES (?, ?, 'STOCK_IN', ?, 0, ?, 'Initial Inventory Setup', ?)
            """, (pid, pname, pstock, pstock, now_str))

    # 2. Seed Customers if empty
    cur.execute("SELECT COUNT(*) FROM customers")
    if cur.fetchone()[0] == 0:
        sample_customers = [
            ("Walk-in Customer", "9999999999", "walkin@market.local", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
            ("Ramesh Kumar", "9876543210", "ramesh@example.com", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
            ("Priya Sharma", "9811223344", "priya@example.com", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
            ("Amit Patel", "9988776655", "amit@example.com", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
        ]
        cur.executemany("""
            INSERT INTO customers (name, phone, email, created_at)
            VALUES (?, ?, ?, ?)
        """, sample_customers)

    # 3. Seed Past Sales if empty
    cur.execute("SELECT COUNT(*) FROM sales")
    if cur.fetchone()[0] == 0:
        sample_sales_data = [
            ("INV-DEMO-01", "Ramesh Kumar", "9876543210", 315.0, 0.0, 0.0, 5.0, 15.75, 330.75, "UPI / QR Code", "Admin", (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d 10:30:00")),
            ("INV-DEMO-02", "Priya Sharma", "9811223344", 405.0, 5.0, 20.25, 5.0, 19.24, 403.99, "Cash", "Admin", (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d 14:15:00")),
            ("INV-DEMO-03", "Walk-in Customer", "9999999999", 260.0, 0.0, 0.0, 5.0, 13.0, 273.0, "Cash", "Cashier", datetime.now().strftime("%Y-%m-%d 11:20:00"))
        ]
        for inv, cname, cphone, sub, disc_p, disc_a, gst_p, gst_a, total, pay, cashier, dt in sample_sales_data:
            cur.execute("""
                INSERT INTO sales (invoice_no, customer_name, customer_phone, subtotal, discount_pct, discount_amt, gst_pct, gst_amt, grand_total, payment_method, cashier, sold_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (inv, cname, cphone, sub, disc_p, disc_a, gst_p, gst_a, total, pay, cashier, dt))
            sale_id = cur.lastrowid

            cur.execute("""
                INSERT INTO sale_items (sale_id, product_id, sku, product_name, cost_price, unit_price, qty, line_total)
                VALUES (?, 1, 'SKU-1001', 'Basmati Rice 1kg', 75.0, 95.0, 2, 190.0)
            """, (sale_id,))

    con.commit()
    con.close()
    print("Demo data seeded successfully.")

seed_sample_data()

# Display current database status
con = sqlite3.connect(DB_NAME)
cur = con.cursor()
cur.execute("SELECT COUNT(*) FROM products")
p_cnt = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM customers")
c_cnt = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM sales")
s_cnt = cur.fetchone()[0]
con.close()
print(f"Database Status: {p_cnt} Products, {c_cnt} Customers, {s_cnt} Sales records.")

## 🧾 Step 4: Invoice & Receipt Generation Engine

The function below automatically handles generating either:
1. **Professional PDF Invoices** (using `reportlab` if installed) saved in the `invoices/` directory.
2. **Clean Text Receipts (`.txt`)** with standard store headers, itemization, GST/discount breakdown, and store policies.

In [ ]:
def generate_invoice_file(sale_id, db_path=DB_NAME):
    """
    Generate PDF invoice (if reportlab installed) or structured text receipt.
    Returns the generated file path.
    """
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("SELECT * FROM sales WHERE id = ?", (sale_id,))
    sale = cur.fetchone()
    if not sale:
        con.close()
        return None

    invoice_no = sale[1]
    cust_name = sale[3] or "Walk-in Customer"
    cust_phone = sale[4] or "N/A"
    subtotal = sale[5]
    discount_pct = sale[6]
    discount_amt = sale[7]
    gst_pct = sale[8]
    gst_amt = sale[9]
    grand_total = sale[10]
    payment_method = sale[11]
    cashier = sale[12]
    sold_at = sale[13]

    cur.execute("SELECT sku, product_name, unit_price, qty, line_total FROM sale_items WHERE sale_id = ?", (sale_id,))
    items = cur.fetchall()
    con.close()

    os.makedirs(INVOICE_DIR, exist_ok=True)

    if REPORTLAB_AVAILABLE:
        pdf_path = os.path.join(INVOICE_DIR, f"{invoice_no}.pdf")
        doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36)
        story = []
        styles = getSampleStyleSheet()

        title_style = ParagraphStyle(
            'InvoiceTitle',
            parent=styles['Heading1'],
            fontSize=20,
            leading=24,
            textColor=colors.HexColor('#1E3A8A'),
            alignment=1
        )
        subtitle_style = ParagraphStyle(
            'InvoiceSub',
            parent=styles['Normal'],
            fontSize=10,
            textColor=colors.gray,
            alignment=1
        )

        story.append(Paragraph("<b>APNA MARKET - POS & INVENTORY</b>", title_style))
        story.append(Paragraph("Main Market Road, City Center | GSTIN: 22AAAAA0000A1Z5 | Phone: +91 98765 00000", subtitle_style))
        story.append(Spacer(1, 15))

        meta_data = [
            [f"Invoice No: {invoice_no}", f"Date: {sold_at}"],
            [f"Customer: {cust_name} ({cust_phone})", f"Payment Mode: {payment_method}"],
            [f"Billed By: {cashier}", "Status: PAID"]
        ]
        meta_table = Table(meta_data, colWidths=[270, 270])
        meta_table.setStyle(TableStyle([
            ('TEXTCOLOR', (0,0), (-1,-1), colors.HexColor('#333333')),
            ('FONTNAME', (0,0), (-1,-1), 'Helvetica-Bold'),
            ('FONTSIZE', (0,0), (-1,-1), 9),
            ('BOTTOMPADDING', (0,0), (-1,-1), 4),
        ]))
        story.append(meta_table)
        story.append(Spacer(1, 10))

        table_data = [["SKU", "Item Description", "Unit Price (₹)", "Qty", "Total (₹)"]]
        for sku, name, price, qty, ltotal in items:
            table_data.append([sku, name, f"{price:.2f}", str(qty), f"{ltotal:.2f}"])

        items_table = Table(table_data, colWidths=[90, 230, 80, 50, 90])
        items_table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1E3A8A')),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('ALIGN', (2, 0), (-1, -1), 'RIGHT'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, 0), 10),
            ('BOTTOMPADDING', (0, 0), (-1, 0), 6),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.lightgrey),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor('#F8FAFC')]),
        ]))
        story.append(items_table)
        story.append(Spacer(1, 10))

        summary_data = [
            ["Subtotal:", f"₹ {subtotal:.2f}"],
            [f"Discount ({discount_pct:.1f}%):", f"- ₹ {discount_amt:.2f}"],
            [f"GST ({gst_pct:.1f}%):", f"+ ₹ {gst_amt:.2f}"],
            ["Grand Total:", f"₹ {grand_total:.2f}"]
        ]
        summary_table = Table(summary_data, colWidths=[400, 140])
        summary_table.setStyle(TableStyle([
            ('ALIGN', (0, 0), (-1, -1), 'RIGHT'),
            ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
            ('FONTSIZE', (0, 0), (-1, -1), 10),
            ('FONTNAME', (0, -1), (-1, -1), 'Helvetica-Bold'),
            ('FONTSIZE', (0, -1), (-1, -1), 12),
            ('TEXTCOLOR', (0, -1), (-1, -1), colors.HexColor('#1E3A8A')),
            ('LINEABOVE', (0, -1), (-1, -1), 1, colors.HexColor('#1E3A8A')),
            ('TOPPADDING', (0, 0), (-1, -1), 3),
            ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
        ]))
        story.append(summary_table)
        story.append(Spacer(1, 20))

        footer_text = "Thank you for your business! Goods once sold cannot be taken back without original bill."
        story.append(Paragraph(footer_text, subtitle_style))

        doc.build(story)
        return pdf_path

    else:
        # Clean formatted text invoice
        txt_path = os.path.join(INVOICE_DIR, f"{invoice_no}.txt")
        lines = []
        lines.append("=" * 55)
        lines.append("           APNA MARKET - POS & INVENTORY")
        lines.append("   Main Market Road, City Center | Ph: +91 98765 00000")
        lines.append("=" * 55)
        lines.append(f"Invoice No : {invoice_no}")
        lines.append(f"Date & Time: {sold_at}")
        lines.append(f"Customer   : {cust_name} ({cust_phone})")
        lines.append(f"Payment    : {payment_method}  | Billed By: {cashier}")
        lines.append("-" * 55)
        lines.append(f"{'Item':<24}{'Price':>8}{'Qty':>6}{'Total':>10}")
        lines.append("-" * 55)
        for sku, name, price, qty, ltotal in items:
            item_display = (name[:22] + '..') if len(name) > 24 else name
            lines.append(f"{item_display:<24}{price:>8.2f}{qty:>6}{ltotal:>10.2f}")
        lines.append("-" * 55)
        lines.append(f"{'Subtotal:':<38} ₹{subtotal:>10.2f}")
        if discount_amt > 0:
            lines.append(f"{'Discount (' + str(discount_pct) + '%):':<38}-₹{discount_amt:>10.2f}")
        if gst_amt > 0:
            lines.append(f"{'GST (' + str(gst_pct) + '%):':<38}+₹{gst_amt:>10.2f}")
        lines.append("=" * 55)
        lines.append(f"{'GRAND TOTAL:':<38} ₹{grand_total:>10.2f}")
        lines.append("=" * 55)
        lines.append("      Thank you for shopping with us! Visit again!   ")
        lines.append("=" * 55)

        with open(txt_path, "w", encoding="utf-8") as f:
            f.write("\n".join(lines))
        return txt_path

# Test invoice generation on demo sale 1
sample_file = generate_invoice_file(1)
print(f"Generated invoice test file: {sample_file}")

## 🖥️ Step 5: The Complete POS & Inventory Desktop Application Class (`MarketApp`)

Below is the complete Tkinter desktop application architecture integrating all 6 core screens and user role permission gating.

In [ ]:
# Load full application class from market_app.py
from market_app import MarketApp

print("✅ MarketApp class loaded successfully!")
print("Available features in MarketApp:")
print(" - Dashboard: Today's Revenue, Units Sold, Low Stock Alerts, Recent Activity")
print(" - Products: Full CRUD, SKU/Barcode, Cost & Selling Prices, Category, Stock Level")
print(" - Billing / POS: Barcode Scan, Product search, Cart table, Discount, GST, Invoice")
print(" - Inventory: Stock in/out adjustments, reason tracking, full audit log")
print(" - Customers: Customer records, phone lookup, full purchase history")
print(" - Reports: Daily/weekly revenue, COGS, estimated profit, best-selling products")
print(" - Roles: Admin / Owner vs Cashier permissions")

## 🚀 Step 6: Launch the Desktop Application

Run the cell below to launch the **Tkinter Desktop Window** on your screen.

> **Note for Jupyter Notebook users**:
> - When running this cell on Windows, the desktop GUI window pops up immediately.
> - You can interact with all screens, add products, scan items, and complete sales.
> - When done, simply close the GUI window to complete cell execution.

In [ ]:
# Initialize database & launch the desktop app
init_db()
seed_sample_data()

# Create and start GUI mainloop
app = MarketApp()
app.mainloop()

## 💻 Step 7: Running as a Standalone Python Script

You can also run the application directly from your terminal or command prompt without opening Jupyter Notebook:

```bash
# Navigate to this project folder
cd "path/to/folder"

# Run the desktop application
python market_app.py
```

### Optional Upgrades & Next Steps:
- **Barcode Scanner Hardware**: Standard USB/Bluetooth barcode scanners act as keyboard input; they automatically type into the `Barcode / SKU Scan` field and press Enter.
- **Thermal Printer Support**: Print directly to ESC/POS 58mm or 80mm thermal receipt printers via `win32print` or `python-escpos`.
- **Export to Excel / CSV**: Use Python's built-in `csv` module to export inventory reports and sales journals for tax/accounting filings.